# 03 — Language modeling: n-grams, smoothing, interpolation, perplexity, generation

## Mental model

A language model assigns probability to a sequence.

```text
P(w1, ..., wn) = product P(wi | previous words)
```

N-gram models approximate this by looking only at the previous `n-1` words. They are old but excellent for understanding probability, smoothing, perplexity, and decoding.

In [ ]:
from pathlib import Path
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
print("Project root:", ROOT)

In [ ]:
import math
import random
from collections import Counter, defaultdict
import pandas as pd

df = pd.read_csv(DATA_DIR / "sample_customer_tickets.csv")
sentences = df["text"].tolist()
sentences[:3]

In [ ]:
import re
TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|\d+(?:\.\d+)?|[^\w\s]")

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

def add_boundaries(tokens, n):
    return ["<s>"] * (n - 1) + tokens + ["</s>"]

[tokenize(s) for s in sentences[:2]]

## Train an n-gram model

We store counts for contexts and next words.

In [ ]:
class NGramLM:
    def __init__(self, n=2, alpha=1.0):
        self.n = n
        self.alpha = alpha
        self.context_counts = Counter()
        self.ngram_counts = Counter()
        self.vocab = set()

    def fit(self, texts):
        for text in texts:
            tokens = tokenize(text)
            self.vocab.update(tokens)
            seq = add_boundaries(tokens, self.n)
            for i in range(self.n - 1, len(seq)):
                context = tuple(seq[i - self.n + 1:i])
                word = seq[i]
                self.context_counts[context] += 1
                self.ngram_counts[(context, word)] += 1
        self.vocab.update(["</s>", "<unk>"])
        return self

    def prob(self, word, context):
        # Add-k smoothing
        if word not in self.vocab:
            word = "<unk>"
        context = tuple(context[-(self.n - 1):]) if self.n > 1 else tuple()
        numerator = self.ngram_counts[(context, word)] + self.alpha
        denominator = self.context_counts[context] + self.alpha * len(self.vocab)
        return numerator / denominator

    def sentence_logprob(self, text):
        tokens = tokenize(text)
        seq = add_boundaries(tokens, self.n)
        total = 0.0
        for i in range(self.n - 1, len(seq)):
            context = tuple(seq[i - self.n + 1:i])
            word = seq[i]
            total += math.log(self.prob(word, context))
        return total

    def perplexity(self, texts):
        total_logprob = 0.0
        total_tokens = 0
        for text in texts:
            tokens = tokenize(text) + ["</s>"]
            total_tokens += len(tokens)
            total_logprob += self.sentence_logprob(text)
        return math.exp(-total_logprob / max(total_tokens, 1))

bigram = NGramLM(n=2, alpha=1.0).fit(sentences)
trigram = NGramLM(n=3, alpha=1.0).fit(sentences)
print("P(crashes | app) =", bigram.prob("crashes", ["app"]))
print("Bigram perplexity:", bigram.perplexity(sentences))
print("Trigram perplexity:", trigram.perplexity(sentences))

## Interpolation

Interpolation mixes multiple models. It is usually a weighted **sum**, not a weighted product.

```text
P = λ1 P_trigram + λ2 P_bigram + λ3 P_unigram
```

In [ ]:
class InterpolatedLM:
    def __init__(self, models, weights):
        assert abs(sum(weights) - 1.0) < 1e-9
        self.models = models
        self.weights = weights

    def prob(self, word, context):
        return sum(w * m.prob(word, context) for w, m in zip(self.weights, self.models))

    def sentence_logprob(self, text):
        tokens = tokenize(text)
        n = max(m.n for m in self.models)
        seq = add_boundaries(tokens, n)
        total = 0.0
        for i in range(n - 1, len(seq)):
            context = tuple(seq[:i])
            word = seq[i]
            total += math.log(self.prob(word, context))
        return total

unigram = NGramLM(n=1, alpha=1.0).fit(sentences)
interp = InterpolatedLM([trigram, bigram, unigram], weights=[0.5, 0.35, 0.15])
for w in ["crashes", "excellent", "banana"]:
    print(w, interp.prob(w, ["the", "app"]))

## Generate text

Sampling from an n-gram model shows the same decoding idea used by neural LMs: choose a next token from a probability distribution.

In [ ]:
def generate(model, max_tokens=20):
    context = ["<s>"] * (model.n - 1)
    output = []
    vocab = sorted(model.vocab)
    for _ in range(max_tokens):
        probs = [model.prob(w, context) for w in vocab]
        total = sum(probs)
        probs = [p / total for p in probs]
        word = random.choices(vocab, weights=probs, k=1)[0]
        if word == "</s>":
            break
        if word != "<unk>":
            output.append(word)
        context.append(word)
    return " ".join(output)

for _ in range(5):
    print(generate(bigram))

## Product relevance

N-gram models are rarely the best production choice today, but the concepts are foundational:

- smoothing prevents zero probability failures
- perplexity measures model surprise
- decoding controls generated outputs
- simple LMs can power autocomplete in constrained domains